# Notebook 03 — Model Training
**DSP501 — Speaker Identification**

This notebook:
- Loads pre-extracted MFCC features
- Trains SVM with GridSearchCV for Pipeline A and B
- Shows CV accuracy scores per fold
- Saves trained models to `models/`

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import numpy as np
import joblib
import matplotlib.pyplot as plt

from train import train_svm, run_experiment, save_results
from evaluation import compute_ci, paired_ttest

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load features

> Make sure you ran `02_features.ipynb` first.

In [ ]:
X_raw  = np.load('../features/features_mfcc_raw.npy')
X_filt = np.load('../features/features_mfcc_filt.npy')
y      = np.load('../features/labels.npy')

print(f'X_raw  : {X_raw.shape}')
print(f'X_filt : {X_filt.shape}')
print(f'y      : {y.shape}  — classes: {sorted(set(y.tolist()))}')

## 2. Train — Experiment A1 (SVM on raw MFCC)

In [ ]:
res_a1, model_a1 = run_experiment('A1_SVM_raw', X_raw, y)

## 3. Train — Experiment B1 (SVM on filtered MFCC)

In [ ]:
res_b1, model_b1 = run_experiment('B1_SVM_filt', X_filt, y)

## 4. CV Scores per Fold

In [ ]:
folds = list(range(1, 6))

plt.figure(figsize=(7, 4))
plt.plot(folds, res_a1['cv_scores'], 'o-', label='Pipeline A (Raw)')
plt.plot(folds, res_b1['cv_scores'], 's-', label='Pipeline B (Filtered)', color='orange')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Cross-Validation Accuracy per Fold')
plt.ylim(0, 1.05)
plt.xticks(folds)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/cv_scores.png', dpi=150)
plt.show()

print(f"A1 — mean: {res_a1['accuracy']['mean']:.4f}  ci95: {res_a1['accuracy']['ci_95']}")
print(f"B1 — mean: {res_b1['accuracy']['mean']:.4f}  ci95: {res_b1['accuracy']['ci_95']}")

## 5. Paired t-test: Does filtering help?

In [ ]:
t_stat, p_value = paired_ttest(res_a1['cv_scores'], res_b1['cv_scores'])

print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_value:.4f}')
print()
if p_value < 0.05:
    print('✓ p < 0.05 — Filtering significantly improves accuracy.')
else:
    print('✗ p >= 0.05 — No statistically significant difference.')

## 6. Save models and results

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model_a1, '../models/svm_pipeline_a.pkl')
joblib.dump(model_b1, '../models/svm_pipeline_b.pkl')
print('Models saved.')

results = {
    'random_seed': 42,
    'cv_folds': 5,
    'experiments': {
        'A1_SVM_raw':  res_a1,
        'B1_SVM_filt': res_b1,
    },
    'statistical_tests': {
        'SVM_A_vs_B': {'t_stat': t_stat, 'p_value': p_value}
    }
}
save_results(results, path='../results.json')